# QFT and phase estimation

Estimate a phase exactly representable with three counting qubits and compare its full output distribution.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [ ]:
from qiskit.circuit.library import QFT

counting = 3
phase = 3 / 8
circuit = QuantumCircuit(counting + 1)
circuit.x(counting)
circuit.h(range(counting))
for wire in range(counting):
    circuit.cp(2 * np.pi * phase * (2 ** wire), wire, counting)
circuit.append(QFT(counting, inverse=True, do_swaps=True).to_gate(), range(counting))

def get_reference():
    return Statevector.from_instruction(circuit).probabilities(qargs=range(counting))

reference, reference_ms, _ = benchmark(get_reference)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def get_mettleq():
    state = backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"]
    return Statevector(state).probabilities(qargs=range(counting))

candidate, mettleq_ms, _ = benchmark(get_mettleq)
error = max_abs_error(reference, candidate)
reference_mode = int(np.argmax(reference))
candidate_mode = int(np.argmax(candidate))
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/08_qft_and_phase_estimation.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase-register probabilities atol=2e-6 and identical mode",
    passed=error <= 2e-6 and candidate_mode == reference_mode,
    exact_match=candidate_mode == reference_mode,
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "expected_phase": phase, "reference_mode": reference_mode, "mettleq_mode": candidate_mode},
)